In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("transformers>=4.40.0", "datasets>=2.19.0", "accelerate>=0.29.0",
    "tqdm", "wandb", "bitsandbytes")

# ── 1. Imports ────────────────────────────────────────────────────────────────
import os, json, math, random
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW

# ── 2. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    base_model        = "EleutherAI/pythia-160m",
    output_dir        = "/kaggle/working/pythia-160m-python-ft",
    dataset_name      = "code_search_net",
    dataset_lang      = "python",
    max_samples_train = 80_000,
    max_samples_eval  = 4_000,
    max_seq_len       = 512,
    epochs            = 1,
    per_device_bs     = 8,
    grad_accum        = 4,
    lr                = 2e-4,
    weight_decay      = 0.01,
    seed              = 50,
    save_steps        = 500,
    eval_steps        = 500,
    logging_steps     = 100,
    cache_activations = True,
    cache_layer       = 6,
    cache_n_tokens    = 200_000,
)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  torch {torch.__version__}")

# ── 3. Load tokenizer & base model ───────────────────────────────────────────
print("\n[1/6] Loading base model …")
tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"])
tokenizer.pad_token = tokenizer.eos_token   # Pythia has no pad token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG["base_model"],
    torch_dtype=torch.float32,
)
base_model.eval()
print(f"  Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
# ── 4. Dataset ────────────────────────────────────────────────────────────────
print("\n[2/6] Loading CodeSearchNet-Python …")
raw = load_dataset(CFG["dataset_name"], CFG["dataset_lang"], trust_remote_code=True)

# We use the "whole_func_string" field which contains the raw Python source
def extract_code(examples):
    return {"text": examples["whole_func_string"]}

train_ds = (
    raw["train"]
    .select(range(min(CFG["max_samples_train"], len(raw["train"]))))
    .map(extract_code, remove_columns=raw["train"].column_names, num_proc=2)
)
eval_ds = (
    raw["validation"]
    .select(range(min(CFG["max_samples_eval"], len(raw["validation"]))))
    .map(extract_code, remove_columns=raw["validation"].column_names, num_proc=2)
)
print(f"  Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")

# ── 5. Tokenise ───────────────────────────────────────────────────────────────

print("\n[3/6] Tokenising …")

def tokenise(examples):
    out = tokenizer(
        examples["text"],
        truncation=True,
        max_length=CFG["max_seq_len"],
        padding="max_length",
    )
    return out

train_tok = train_ds.map(tokenise, batched=True, remove_columns=["text"], num_proc=2)
eval_tok  = eval_ds.map(tokenise,  batched=True, remove_columns=["text"], num_proc=2)
train_tok.set_format("torch")
eval_tok.set_format("torch")

In [ ]:
# ── 6. Fine-tune ──────────────────────────────────────────────────────────────
print("\n[4/6] Fine-tuning …")

# 1. Force the model to load in pure FP32
ft_model = AutoModelForCausalLM.from_pretrained(
    CFG["base_model"],
    torch_dtype=torch.float32,   # <-- Forcing pure FP32 here
    device_map="auto",
)

# 2. Configure TrainingArguments strictly without FP16
training_args = TrainingArguments(
    output_dir                  = CFG["output_dir"],
    num_train_epochs            = CFG["epochs"],
    per_device_train_batch_size = CFG["per_device_bs"],
    per_device_eval_batch_size  = CFG["per_device_bs"],
    gradient_accumulation_steps = CFG["grad_accum"],
    learning_rate               = CFG["lr"],
    warmup_steps                = 100,
    optim                       = "paged_adamw_8bit",  # Keeps memory usage low despite FP32 weights
    weight_decay                = CFG["weight_decay"],
    eval_strategy               = "steps",
    eval_steps                  = CFG["eval_steps"],
    save_strategy               = "steps",
    save_steps                  = CFG["save_steps"],
    logging_steps               = CFG["logging_steps"],
    load_best_model_at_end      = True,
    seed                        = CFG["seed"],
    report_to                   = "none",
    dataloader_num_workers      = 2,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model           = ft_model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = eval_tok,
    data_collator   = collator,
)


trainer.train()
# Move entire model to CPU before saving
ft_model = ft_model.cpu()
trainer.model = ft_model

trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print(f"✓ Saved to {CFG['output_dir']}")

# Verify model is clean before saving
test_ids = tokenizer("def hello():", return_tensors="pt")["input_ids"]
with torch.no_grad():
    test_out = ft_model(test_ids)
print(f"Logits NaN: {torch.isnan(test_out.logits).sum().item()}")
print(f"Logits mean: {test_out.logits.mean():.4f}")
assert torch.isnan(test_out.logits).sum() == 0, "Model has NaN weights — do not save!"
print("✓ Model is clean, safe to save")
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print(f"  ✓ Fine-tuned model saved to {CFG['output_dir']}")

In [ ]:
# ── 7. Quick perplexity sanity check ─────────────────────────────────────────
print("\n[5/6] Perplexity check …")

def compute_perplexity(model, dataset, n=200):
    model.eval()
    losses = []
    with torch.no_grad():
        for i in range(min(n, len(dataset))):
            ids    = dataset[i]["input_ids"].unsqueeze(0).to(device)
            labels = dataset[i]["labels"].unsqueeze(0).to(device)
            out    = model(ids, labels=labels)
            losses.append(out.loss.item())
    return math.exp(np.mean(losses))

ft_model.to(device)
base_model.to(device)
ppl_ft   = compute_perplexity(ft_model,   eval_tok)
ppl_base = compute_perplexity(base_model, eval_tok)
print(f"  Base model PPL on Python eval : {ppl_base:.2f}")
print(f"  Fine-tuned PPL on Python eval : {ppl_ft:.2f}")

# ── 8. Cache activations from the fine-tuned model ───────────────────────────
if CFG["cache_activations"]:
    print(f"\n[6/6] Caching layer-{CFG['cache_layer']} activations ({CFG['cache_n_tokens']:,} tokens) …")

    cache_path = Path("/kaggle/working/act_cache_finetuned.npy")
    act_list   = []
    total_tok  = 0

    def hook_fn(module, inp, out):
        # out is a tuple — first element is the hidden state [B, T, D]
        act_list.append(out[0].detach().cpu().float())

    hook_handle = ft_model.gpt_neox.layers[CFG["cache_layer"]].register_forward_hook(hook_fn)

    ft_model.eval().to(device)
    with torch.no_grad():
        for i in tqdm(range(len(train_tok))):
            if total_tok >= CFG["cache_n_tokens"]:
                break
            ids = train_tok[i]["input_ids"].unsqueeze(0).to(device)
            ft_model(ids)
            total_tok += ids.shape[1]

    hook_handle.remove()

    acts = torch.cat(act_list, dim=1).squeeze(0).numpy()
    acts = acts[:CFG["cache_n_tokens"]]

    # Verify clean before saving
    print(f"  NaN count : {np.isnan(acts).sum()}")
    print(f"  Mean      : {np.nanmean(acts):.4f}")
    print(f"  Std       : {np.nanstd(acts):.4f}")

    np.save(cache_path, acts)
    print(f"  ✓ Saved activations: shape {acts.shape}  →  {cache_path}")

# ── 9. Save config ────────────────────────────────────────────────────────────
with open("/kaggle/working/finetune_cfg.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("\n✅ Notebook 1 complete. Outputs in /kaggle/working/")
print("   ├── pythia-160m-python-ft/   (model weights)")
print("   ├── act_cache_finetuned.npy  (layer activations)")
print("   └── finetune_cfg.json        (shared config)")